In [ ]:
### Document Structure

from langchain_core.documents import Document




In [ ]:
doc = Document(
    page_content = "This is the content of the document.",
    metadata = {
        "source":"example.pdf",
        "pages": 10,
        "author":"Zayed Shah",
        "date_created":"2024-06-01"
    }
)

doc

In [ ]:
## Create a simple txt file

import os 

os.makedirs("../data/text_files",exist_ok=True)

In [ ]:
sample_texts = {
    "../data/text_files/python_intro.txt": """ 

    Python is a high-level, interpreted programming language known for its simplicity and readability. 
    It was created by Guido van Rossum and first released in 1991. 
    Python's design philosophy emphasizes code readability and a syntax that allows programmers to express concepts in fewer lines of code compared to other programming languages.
    Some of the use cases of Python include web development, data analysis, artificial intelligence, scientific computing, automation, and more.
    Apart from its versatility, Python has a large and active community that contributes to a vast ecosystem of libraries and frameworks, making it a popular choice for developers across various domains.
    Python's ease of use and extensive libraries have made it a go-to language for beginners and experienced programmers alike, solidifying its position as one of the most widely used programming languages in the world.
    """,
    "../data/text_files/data_science.txt": """
    Data science is an interdisciplinary field that uses scientific methods, processes, algorithms and systems to extract knowledge and insights from structured and unstructured data.
    It involves statistics, machine learning, data visualization, and domain expertise to solve complex problems and make data-driven decisions.
    Data scientists analyze and interpret complex data to help organizations make informed decisions, identify trends, and gain a competitive edge in various industries such as finance, healthcare, marketing, and more.
    Data science combines techniques from various fields, including mathematics, computer science, and domain-specific knowledge, to extract valuable insights from data and drive innovation in today's data-driven world.
    Machine learning, a subset of data science, enables computers to learn from data and make predictions or decisions without being explicitly programmed, further enhancing the capabilities of data science in solving real-world problems.
    """
}

for filepath, content in sample_texts.items():
    with open(filepath,"w", encoding="utf-8") as f:
        f.write(content)

print("Sample text files created successfully.")

In [ ]:
# Text Loader

# from langchain.document_loaders import TextLoader
from langchain_community.document_loaders import TextLoader 


loader = TextLoader("../data/text_files/python_intro.txt",encoding="utf-8")
document  = loader.load()
print(document)


In [ ]:
# Directory Loader

from langchain_community.document_loaders import DirectoryLoader

dir_loader = DirectoryLoader(
    "../data/text_files",
    glob = "**/*.txt",  #pattern to match
    loader_cls = TextLoader, #loader class to use
    loader_kwargs={'encoding':"utf-8"},
    show_progress=False
)

print(dir_loader.load())

In [ ]:

from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader

dir_loader = DirectoryLoader(
    "../data/pdf",
    glob = "**/*.pdf",  #pattern to match
    loader_cls = PyMuPDFLoader, #loader class to use
    show_progress=False
)

pdf_documents = dir_loader.load()
pdf_documents

In [ ]:
#  Splitting the pdf into chunks

from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """
    Split documents into smaller chunks for better RAG performance.
    
    Parameters:
    - chunk_size: Maximum characters per chunk (adjust based on your LLM)
    - chunk_overlap: Characters to overlap between chunks (preserves context)
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, # Each chunk: ~1000 characters
        chunk_overlap=chunk_overlap, # 200 chars overlap for context
        length_function=len, # How to measure length
        separators=["\n\n", "\n", " ", ""] # Split hierarchy
    )
    # Actually split the documents
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show what a chunk looks like
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [ ]:
chunks = split_documents(pdf_documents)
chunks

#  Embedding and VectorStore DB

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid 
from typing import List, Dict, Any, Tuple 
from sklearn.metrics.pairwise import cosine_similarity


In [ ]:
class EmbeddingManager:
    """ Handles document embedding generation using SenctenceTransfromer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
            Initialize Embedding Manager
        """

        self.model_name = model_name
        self.model = None
        self._load_model()
    
    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading model: {self.model_name}...")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding Dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model: {self.model_name}: {e}")
            raise
    
    def generate_embedding(self, texts: List[str]) -> np.ndarray:
        """ Generate Embedding for a list of texts

            Args: lists of txt strings to embed

            Returns: numpy array of embeddings
        """
        if not self.model:
            raise ValueError("Model not loaded. Cannot generate embeddings.")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Embeddings generated successfully with shape: {embeddings.shape}.")
        return embeddings
    

# Initialize the embedding manager

embedding_manager = EmbeddingManager()
embedding_manager

        

#  Vector Store

In [ ]:
class VectorStore:
    """Manages document embeddings in ChromaDB"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """ 
        Initialize the Vector Store
        Args:
            collection_name: Name of the ChromaDB collection to store embeddings
            persist_directory: Directory where the ChromaDB database will be persisted
        """

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None 
        self.initialize_store()

    def initialize_store(self):
        """ Initialize ChromaDB client and collection"""

        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client  = chromadb.PersistentClient(path=self.persist_directory)

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata = {"description": "Collection of PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized successfully with collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self,documents: List[Any], embeddings: np.ndarray):
        """
            Add documents and embeddings to vector store
            Args:
                documents: List of document metadata or content to store
                embeddings: Corresponding list of embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must match.")
        
        print(f"Adding {len(documents)} documents to vector store...")

        ids=[]
        metadatas = []
        document_text=[]
        embeddings_list=[]

        for i, (doc,embedding) in enumerate(zip(documents,embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            document_text.append(doc.page_content) 
            embeddings_list.append(embedding.tolist())
        
        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=document_text,
                embeddings=embeddings_list
            )
            print(f"Documents added successfully. Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore = VectorStore()
vectorstore

In [ ]:
#  Convert the text to embeddings

texts = [doc.page_content for doc in chunks]
# texts 

embeddings = embedding_manager.generate_embedding(texts)

vectorstore.add_documents(chunks,embeddings)

## Retrieval

In [ ]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self,vectorstore: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever 

        Args:
            vectorstore: Instance of the VectorStore class to retrieve documents from
            embedding_manager: Instance of the EmbeddingManager to generate query embeddings 
        """
        self.vectorstore = vectorstore
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = -0.5) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents based on the query

        Args:
            query: The user query string to search for
            top_k: Number of top relevant documents to retrieve
        Returns:
            List of retrieved documents with metadata
        """
        print(f"Retrieving documents for query: '{query}' with top_k={top_k}...")
        query_embedding = self.embedding_manager.generate_embedding([query])[0]

        try:
            results = self.vectorstore.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
            )
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids,documents, metadatas, distances)):
                    similarity_score = 1 - distance

                    if similarity_score  >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i+1
                        })
            print(f"Retrieved {len(retrieved_docs)} relevant documents.")
            return retrieved_docs
        except Exception as e:
            print(f"Error during retrieval: {e}")
            raise
                        

rag_retriever = RAGRetriever(vectorstore, embedding_manager)
rag_retriever

            
           


In [ ]:
results = rag_retriever.retrieve("Tell about the offer letter", score_threshold=-1.0)
for doc in results[:3]:  # Show first 3 results
    print(f"Similarity: {doc['similarity_score']:.3f}")
    print(f"Content: {doc['content'][:200]}...")
    print("---")